In [ ]:
!pip install -q pandas numpy scikit-learn mlflow fastapi uvicorn joblib pytest nest-asyncio httpx

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

from IPython.display import display

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
folders = [
    "data/raw",
    "data/feature_store",
    "models",
    "tests"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [ ]:
def generate_milk_data(n=250, summer=False, seed=42):

    np.random.seed(seed + (100 if summer else 0))

    data = pd.DataFrame({
        "milk_quantity_litres": np.random.uniform(20, 500, n),

        "collection_temperature_c": np.random.normal(
            5.5 if not summer else 8, 2, n
        ).clip(0, 16),

        "transport_duration_hours": np.random.normal(
            1.8 if not summer else 2.8, 0.7 if not summer else 1, n
        ).clip(0.2, 6),

        "cooling_delay_hours": np.random.gamma(
            1.8, 0.35, n
        ) + (0.25 if summer else 0),

        "milk_fat_percentage": np.random.normal(
            4, 0.55, n
        ).clip(2.2, 6),

        "acidity_indicator": np.random.normal(
            0.16, 0.035, n
        ).clip(0.08, 0.32),

        "container_type": np.random.choice(
            ["Stainless Steel", "Food Grade Plastic", "Aluminium"],
            n
        ),

        "ambient_temperature_c": np.random.normal(
            28 if not summer else 36,
            4 if not summer else 3,
            n
        ).clip(18, 42)
    })

    score = (
        -3
        + 0.22 * data["collection_temperature_c"]
        + 0.55 * data["transport_duration_hours"]
        + 1.15 * data["cooling_delay_hours"]
        + 13 * (data["acidity_indicator"] - 0.16)
        + 0.045 * (data["ambient_temperature_c"] - 28)
        + np.where(data["container_type"] == "Aluminium", 0.25, 0)
    )

    probability = 1 / (1 + np.exp(-score))

    data["batch_rejected"] = np.where(
        np.random.binomial(1, probability) == 1,
        "Yes",
        "No"
    )

    return data


df_v1 = generate_milk_data(250, summer=False)

df_v1.to_csv("data/raw/milk_batches_v1.csv", index=False)

print("Version 1 dataset created!")
print("Number of records:", len(df_v1))

display(df_v1.head(10))

Version 1 dataset created!
Number of records: 250


,milk_quantity_litres,collection_temperature_c,transport_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected
0,199.779257,3.806413,0.607632,0.877506,4.887541,0.235506,Food Grade Plastic,31.627431,Yes
1,476.342867,2.470306,1.761117,0.725609,4.249444,0.138800,Aluminium,27.138007,No
2,371.357092,4.606970,2.068846,0.177926,3.865714,0.185973,Aluminium,31.226257,No
3,307.356072,7.212798,1.777114,0.821172,4.530248,0.170475,Stainless Steel,28.824198,Yes
4,94.888947,5.928187,0.352791,0.664352,4.654209,0.205561,Aluminium,27.259339,No
5,94.877370,3.008522,1.737616,0.144677,3.324816,0.214653,Aluminium,21.628789,Yes
6,47.880134,5.846362,0.886871,0.322115,4.328570,0.161120,Food Grade Plastic,22.340190,No
7,435.764550,6.270635,2.268771,1.098716,4.385645,0.133630,Aluminium,18.000000,Yes
8,308.535206,3.732285,2.056619,0.095844,3.836340,0.176099,Food Grade Plastic,25.749546,Yes
9,359.874837,5.807450,1.142084,0.815431,4.756639,0.136280,Stainless Steel,37.112995,Yes


In [ ]:
df_v2 = generate_milk_data(
    n=250,
    summer=True,
    seed=42
)

df_v2.to_csv(
    "data/raw/milk_batches_v2_summer.csv",
    index=False
)

print("Summer Version 2 created!")
display(df_v2.head(10))

Summer Version 2 created!


,milk_quantity_litres,collection_temperature_c,transport_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected
0,452.989532,8.152153,1.542079,0.296629,3.225643,0.171745,Aluminium,30.037296,Yes
1,287.747617,4.500250,2.327847,0.852999,3.047919,0.189037,Food Grade Plastic,37.207980,No
2,334.872659,10.208145,3.059284,0.711448,4.456541,0.166894,Food Grade Plastic,39.644967,No
3,419.586278,7.020819,4.509486,0.759343,3.587220,0.180860,Food Grade Plastic,31.695224,Yes
4,115.944410,6.941675,2.274900,1.245377,3.824081,0.136238,Stainless Steel,34.546223,Yes
5,81.082044,6.686047,3.534518,1.494697,4.019844,0.104878,Stainless Steel,30.807907,Yes
6,390.290771,5.956230,2.426129,0.438029,4.235296,0.080000,Food Grade Plastic,38.430699,No
7,227.498505,11.048571,2.509420,0.763930,3.676212,0.161366,Stainless Steel,29.532771,Yes
8,204.935471,6.472895,3.853008,1.152662,3.389734,0.151184,Stainless Steel,34.016132,Yes
9,396.148818,10.333800,3.559993,1.741951,3.890188,0.128911,Aluminium,40.221118,Yes


In [ ]:
print("Version 1 shape:", df_v1.shape)
print("Version 2 shape:", df_v2.shape)

print("\nVersion 1 rejection count:")
print(df_v1["batch_rejected"].value_counts())

print("\nVersion 2 rejection count:")
print(df_v2["batch_rejected"].value_counts())

Version 1 shape: (250, 9)
Version 2 shape: (250, 9)

Version 1 rejection count:
batch_rejected
Yes    139
No     111
Name: count, dtype: int64

Version 2 rejection count:
batch_rejected
Yes    213
No      37
Name: count, dtype: int64


In [ ]:
def feature_engineering(df):

    df = df.copy()

    df["temperature_exposure_index"] = (
        df["collection_temperature_c"]
        * df["transport_duration_hours"]
        +
        df["ambient_temperature_c"]
        * df["cooling_delay_hours"]
    )

    df["transport_delay_risk"] = (
        df["transport_duration_hours"]
        *
        (1 + df["cooling_delay_hours"])
    )

    return df


features_v1 = feature_engineering(df_v1)
features_v2 = feature_engineering(df_v2)

features_v1.to_csv(
    "data/feature_store/milk_features_v1.csv",
    index=False
)

features_v2.to_csv(
    "data/feature_store/milk_features_v2_summer.csv",
    index=False
)

print("Feature engineering completed!")

display(features_v1.head())

Feature engineering completed!


,milk_quantity_litres,collection_temperature_c,transport_duration_hours,cooling_delay_hours,milk_fat_percentage,acidity_indicator,container_type,ambient_temperature_c,batch_rejected,temperature_exposure_index,transport_delay_risk
0,199.779257,3.806413,0.607632,0.877506,4.887541,0.235506,Food Grade Plastic,31.627431,Yes,30.066161,1.140833
1,476.342867,2.470306,1.761117,0.725609,4.249444,0.138800,Aluminium,27.138007,No,24.042071,3.038998
2,371.357092,4.606970,2.068846,0.177926,3.865714,0.185973,Aluminium,31.226257,No,15.087086,2.436948
3,307.356072,7.212798,1.777114,0.821172,4.530248,0.170475,Stainless Steel,28.824198,Yes,36.487597,3.236430
4,94.888947,5.928187,0.352791,0.664352,4.654209,0.205561,Aluminium,27.259339,No,20.201204,0.587168


In [ ]:
feature_store = pd.read_csv(
    "data/feature_store/milk_features_v1.csv"
)

print("Reusing stored engineered features")

display(feature_store[
    [
        "temperature_exposure_index",
        "transport_delay_risk",
        "batch_rejected"
    ]
].head(10))

Reusing stored engineered features


,temperature_exposure_index,transport_delay_risk,batch_rejected
0,30.066161,1.140833,Yes
1,24.042071,3.038998,No
2,15.087086,2.436948,No
3,36.487597,3.236430,Yes
4,20.201204,0.587168,No
5,8.356851,1.989010,Yes
6,12.381088,1.172546,No
7,34.003527,4.761506,Yes
8,10.143836,2.253734,Yes
9,36.895691,2.073375,Yes


In [ ]:
df = pd.read_csv(
    "data/feature_store/milk_features_v1.csv"
)

X = df.drop(columns=["batch_rejected"])

y = df["batch_rejected"].map({
    "No": 0,
    "Yes": 1
})

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 187
Testing records: 63


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2
    )
}

results = []
trained_models = {}

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", classifier)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    precision = precision_score(
        y_test, predictions, zero_division=0
    )

    recall = recall_score(
        y_test, predictions, zero_division=0
    )

    f1 = f1_score(
        y_test, predictions, zero_division=0
    )

    accuracy = accuracy_score(y_test, predictions)

    results.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Accuracy": accuracy
    })

    trained_models[name] = pipeline

comparison = pd.DataFrame(results)

display(comparison.round(4))

,Model,Precision,Recall,F1 Score,Accuracy
0,Logistic Regression,0.5758,0.5429,0.5588,0.5238
1,Random Forest,0.5455,0.6857,0.6076,0.5079


In [ ]:
best_model_name = comparison.loc[
    comparison["F1 Score"].idxmax(),
    "Model"
]

best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/best_model.joblib"
)

comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("Best model saved successfully!")

Best model: Random Forest
Best model saved successfully!


In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment(
    "Milk_Batch_Rejection_Prediction"
)

print("MLflow experiment created!")

2026/11/18 09:04:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/18 09:04:38 INFO mlflow.store.db.utils: Updating database tables
2026/11/18 09:04:42 INFO mlflow.tracking.fluent: Experiment with name 'Milk_Batch_Rejection_Prediction' does not exist. Creating a new experiment.


MLflow experiment created!


In [ ]:
with mlflow.start_run(
    run_name=best_model_name
) as run:

    mlflow.log_param(
        "selected_model",
        best_model_name
    )

    best_predictions = best_model.predict(X_test)

    mlflow.log_metric(
        "precision",
        precision_score(y_test, best_predictions, zero_division=0)
    )

    mlflow.log_metric(
        "recall",
        recall_score(y_test, best_predictions, zero_division=0)
    )

    mlflow.log_metric(
        "f1_score",
        f1_score(y_test, best_predictions, zero_division=0)
    )

    # Explicitly trust the required types for ColumnTransformer and RandomForest
    trusted_types = [
        "numpy.dtype",
        "sklearn.compose._column_transformer._RemainderColsList",
        "sklearn.tree._tree.Tree"
    ]

    mlflow.sklearn.log_model(
        best_model,
        artifact_path="model",
        skops_trusted_types=trusted_types
    )

    run_id = run.info.run_id

model_uri = f"runs:/{run_id}/model"

registered_model = mlflow.register_model(
    model_uri,
    "MilkBatchRejectionModel"
)

print("Model registered successfully!")
print("Model name:", registered_model.name)
print("Model version:", registered_model.version)

2026/11/18 09:18:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'MilkBatchRejectionModel'.
2026/11/18 09:18:43 WARNING mlflow.tracking._model_registry.fluent: Run with id 1e33cc9474ec405c8c18ebbf0c14dc24 has no artifacts at artifact path 'model', registering model based on models:/m-e5f6874fd6084a449df69d0c9b120306 instead


Model registered successfully!
Model name: MilkBatchRejectionModel
Model version: 1


Created version '1' of model 'MilkBatchRejectionModel'.


In [ ]:
print("Experiment tracking completed!")

print("Run ID:", run_id)

print("Registered model: MilkBatchRejectionModel")

Experiment tracking completed!
Run ID: 1e33cc9474ec405c8c18ebbf0c14dc24
Registered model: MilkBatchRejectionModel


In [ ]:
!mlflow ui --host 0.0.0.0 --port 5000 --backend-store-uri sqlite:///mlflow.db > mlflow.log 2>&1 &

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

app = FastAPI(
    title="Milk Batch Rejection Prediction API"
)

class MilkBatch(BaseModel):

    milk_quantity_litres: float = Field(gt=0)

    collection_temperature_c: float

    transport_duration_hours: float = Field(ge=0)

    cooling_delay_hours: float = Field(ge=0)

    milk_fat_percentage: float = Field(ge=0)

    acidity_indicator: float

    container_type: str

    ambient_temperature_c: float

In [ ]:
def validate_record(data):

    if data["milk_quantity_litres"] <= 0:
        raise ValueError("Invalid milk quantity")

    if not 0 <= data["collection_temperature_c"] <= 50:
        raise ValueError("Impossible collection temperature")

    if not 0 <= data["ambient_temperature_c"] <= 60:
        raise ValueError("Impossible ambient temperature")

    if not 0 <= data["transport_duration_hours"] <= 24:
        raise ValueError("Invalid transport duration")

    if not 0 <= data["cooling_delay_hours"] <= 24:
        raise ValueError("Invalid cooling delay")

    if not 0 <= data["milk_fat_percentage"] <= 15:
        raise ValueError("Invalid milk fat percentage")

    if not 0 < data["acidity_indicator"] < 1:
        raise ValueError("Invalid acidity indicator")

    if data["container_type"] not in [
        "Stainless Steel",
        "Food Grade Plastic",
        "Aluminium"
    ]:
        raise ValueError("Invalid container type")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Milk Batch Prediction API Running"
    }


@app.post("/predict")
def predict(batch: MilkBatch):

    data = batch.model_dump()

    try:
        validate_record(data)

    except ValueError as e:
        raise HTTPException(
            status_code=422,
            detail=str(e)
        )

    input_df = pd.DataFrame([data])

    input_df = feature_engineering(input_df)

    probability = best_model.predict_proba(
        input_df
    )[0][1]

    prediction = (
        "Rejection Risk"
        if probability >= 0.5
        else "Likely Accepted"
    )

    return {
        "prediction": prediction,
        "rejection_probability": round(float(probability), 4)
    }


print("FastAPI application created!")

FastAPI application created!


In [ ]:
client = TestClient(app)

response = client.get("/")

print(response.json())

{'message': 'Milk Batch Prediction API Running'}


In [ ]:
sample_data = {
    "milk_quantity_litres": 100,
    "collection_temperature_c": 5,
    "transport_duration_hours": 1.5,
    "cooling_delay_hours": 0.3,
    "milk_fat_percentage": 4.1,
    "acidity_indicator": 0.16,
    "container_type": "Stainless Steel",
    "ambient_temperature_c": 29
}

response = client.post(
    "/predict",
    json=sample_data
)

print("Status code:", response.status_code)

print("Prediction:", response.json())

Status code: 200
Prediction: {'prediction': 'Likely Accepted', 'rejection_probability': 0.4032}


In [ ]:
invalid_data = sample_data.copy()

invalid_data["milk_quantity_litres"] = -10

response = client.post(
    "/predict",
    json=invalid_data
)

print("Status code:", response.status_code)

print("Response:", response.json())

Status code: 422
Response: {'detail': [{'type': 'greater_than', 'loc': ['body', 'milk_quantity_litres'], 'msg': 'Input should be greater than 0', 'input': -10, 'ctx': {'gt': 0.0}}]}


In [ ]:
422

422

In [ ]:
invalid_data = sample_data.copy()

invalid_data["collection_temperature_c"] = 90

response = client.post(
    "/predict",
    json=invalid_data
)

print("Status code:", response.status_code)

print("Response:", response.json())

Status code: 422
Response: {'detail': 'Impossible collection temperature'}


In [ ]:
422

422

In [ ]:
print("API title:", app.title)

print("Available API endpoints:")

for route in app.routes:
    print(route.path, route.methods)

API title: Milk Batch Rejection Prediction API
Available API endpoints:
/openapi.json {'GET', 'HEAD'}
/docs {'GET', 'HEAD'}
/docs/oauth2-redirect {'GET', 'HEAD'}
/redoc {'GET', 'HEAD'}
/ {'GET'}
/predict {'POST'}


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
%%writefile validation.py

def validate_record(data):

    if data["milk_quantity_litres"] <= 0:
        raise ValueError("Invalid milk quantity")

    if not 0 <= data["collection_temperature_c"] <= 50:
        raise ValueError("Invalid collection temperature")

    if not 0 <= data["ambient_temperature_c"] <= 60:
        raise ValueError("Invalid ambient temperature")

    if not 0 <= data["transport_duration_hours"] <= 24:
        raise ValueError("Invalid transport duration")

    if not 0 <= data["cooling_delay_hours"] <= 24:
        raise ValueError("Invalid cooling delay")

    if not 0 <= data["milk_fat_percentage"] <= 15:
        raise ValueError("Invalid milk fat")

    if not 0 < data["acidity_indicator"] < 1:
        raise ValueError("Invalid acidity")

    if data["container_type"] not in [
        "Stainless Steel",
        "Food Grade Plastic",
        "Aluminium"
    ]:
        raise ValueError("Invalid container")

Writing validation.py


In [ ]:
%%writefile test_validation.py

import pytest
from validation import validate_record

GOOD_DATA = {
    "milk_quantity_litres": 100,
    "collection_temperature_c": 5,
    "transport_duration_hours": 1.5,
    "cooling_delay_hours": 0.3,
    "milk_fat_percentage": 4.1,
    "acidity_indicator": 0.16,
    "container_type": "Stainless Steel",
    "ambient_temperature_c": 29
}

def test_valid_data():
    validate_record(GOOD_DATA)

def test_negative_quantity():
    data = GOOD_DATA.copy()
    data["milk_quantity_litres"] = -10

    with pytest.raises(ValueError):
        validate_record(data)

def test_impossible_temperature():
    data = GOOD_DATA.copy()
    data["collection_temperature_c"] = 90

    with pytest.raises(ValueError):
        validate_record(data)

def test_invalid_ambient_temperature():
    data = GOOD_DATA.copy()
    data["ambient_temperature_c"] = -10

    with pytest.raises(ValueError):
        validate_record(data)

Writing test_validation.py


In [ ]:
!pytest -v test_validation.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0
collected 4 items                                                              

test_validation.py::test_valid_data PASSED                               [ 25%]
test_validation.py::test_negative_quantity PASSED                        [ 50%]
test_validation.py::test_impossible_temperature PASSED                   [ 75%]
test_validation.py::test_invalid_ambient_temperature PASSED              [100%]

============================== 4 passed in 0.04s ===============================


In [ ]:
# 4 passed

In [ ]:
os.makedirs(
    ".github/workflows",
    exist_ok=True
)

workflow = """
name: Milk Batch MLOps CI

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          pip install pytest pandas numpy

      - name: Run automated tests
        run: |
          pytest -v test_validation.py
"""

with open(
    ".github/workflows/ci.yml",
    "w"
) as f:
    f.write(workflow)

print("GitHub Actions workflow created!")

GitHub Actions workflow created!


In [ ]:
requirements = """
pandas
numpy
scikit-learn
mlflow
fastapi
uvicorn
joblib
pytest
httpx
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Requirements file created!")

Requirements file created!


In [ ]:
readme = """
# Milk Batch Rejection Prediction MLOps Project

## Objective
Predict whether a milk batch is likely to be rejected.

## Dataset
250 synthetic records in Version 1.
250 synthetic records in Summer Version 2.

## Technologies
- Python
- Pandas
- NumPy
- Scikit-learn
- MLflow
- FastAPI
- GitHub Actions

## Feature Engineering
1. Temperature Exposure Index
2. Transport Delay Risk

## Machine Learning Algorithms
1. Logistic Regression
2. Random Forest

## Evaluation Metrics
- Precision
- Recall
- F1-score
- Accuracy

## Deployment
FastAPI prediction endpoint: /predict

## CI/CD
GitHub Actions automated testing.

## Limitation
This project uses synthetic educational data and is not
validated for real-world milk safety decisions.
"""

with open("README.md", "w") as f:
    f.write(readme)

print("README created!")

README created!


In [ ]:
app_code = '''
import joblib
import pandas as pd

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from validation import validate_record

app = FastAPI(title="Milk Batch Prediction API")

model = joblib.load("models/best_model.joblib")

class MilkBatch(BaseModel):
    milk_quantity_litres: float = Field(gt=0)
    collection_temperature_c: float
    transport_duration_hours: float = Field(ge=0)
    cooling_delay_hours: float = Field(ge=0)
    milk_fat_percentage: float = Field(ge=0)
    acidity_indicator: float
    container_type: str
    ambient_temperature_c: float

def engineer_features(df):
    df = df.copy()

    df["temperature_exposure_index"] = (
        df["collection_temperature_c"] *
        df["transport_duration_hours"] +
        df["ambient_temperature_c"] *
        df["cooling_delay_hours"]
    )

    df["transport_delay_risk"] = (
        df["transport_duration_hours"] *
        (1 + df["cooling_delay_hours"])
    )

    return df

@app.post("/predict")
def predict(batch: MilkBatch):
    data = batch.model_dump()

    try:
        validate_record(data)
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    df = engineer_features(pd.DataFrame([data]))

    probability = float(model.predict_proba(df)[0][1])

    return {
        "prediction": (
            "Rejection Risk" if probability >= 0.5
            else "Likely Accepted"
        ),
        "rejection_probability": round(probability, 4)
    }
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [ ]:
import shutil

project_files = [
    "data",
    "models",
    "tests",
    ".github",
    "app.py",
    "validation.py",
    "test_validation.py",
    "requirements.txt",
    "README.md",
    "model_comparison.csv"
]

os.makedirs("milk_batch_mlops_submission", exist_ok=True)

for item in project_files:

    if os.path.exists(item):

        destination = os.path.join(
            "milk_batch_mlops_submission",
            item
        )

        if os.path.isdir(item):
            shutil.copytree(
                item,
                destination,
                dirs_exist_ok=True
            )

        else:
            shutil.copy2(item, destination)

shutil.make_archive(
    "milk_batch_mlops_submission",
    "zip",
    "milk_batch_mlops_submission"
)

print("Complete project ZIP created!")

Complete project ZIP created!


In [ ]:
from google.colab import files

files.download(
    "milk_batch_mlops_submission.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>